<a href="https://colab.research.google.com/github/amritaa72/Runs-and-Wickets/blob/main/WBBL_HTML_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import pandas as pd
import re
import json

from bs4 import BeautifulSoup

# **inspect the HTML structure**

In [2]:
"""
# Inspect the HTML structure around the commentary section

heading = soup.find(
    string=lambda text: text and "Ball-by-ball commentary" in text
)

if heading:
    element = heading.parent

    print("COMMENTARY HEADING ANCESTORS\n")

    for level in range(6):
        if element is None:
            break

        print(f"\nLEVEL {level}")
        print("Tag:", element.name)
        print("Attributes:", element.attrs)
        print("Text:", element.get_text(" ", strip=True)[:500])

        element = element.parent


# Inspect HTML elements containing a delivery number

print("\n\nDELIVERY ELEMENTS\n")

for node in soup.find_all(string=lambda text: text and text.strip() == "16.4")[:5]:
    element = node.parent

    print("\nDelivery candidate:")
    print(element.prettify()[:1500])
    """


'\n# Inspect the HTML structure around the commentary section\n\nheading = soup.find(\n    string=lambda text: text and "Ball-by-ball commentary" in text\n)\n\nif heading:\n    element = heading.parent\n\n    print("COMMENTARY HEADING ANCESTORS\n")\n\n    for level in range(6):\n        if element is None:\n            break\n\n        print(f"\nLEVEL {level}")\n        print("Tag:", element.name)\n        print("Attributes:", element.attrs)\n        print("Text:", element.get_text(" ", strip=True)[:500])\n\n        element = element.parent\n\n\n# Inspect HTML elements containing a delivery number\n\nprint("\n\nDELIVERY ELEMENTS\n")\n\nfor node in soup.find_all(string=lambda text: text and text.strip() == "16.4")[:5]:\n    element = node.parent\n\n    print("\nDelivery candidate:")\n    print(element.prettify()[:1500])\n    '

# **Fetch the match page**

In [3]:
url = "https://cricketsix.com/match/162733/sydney-thunder-women-vs-brisbane-heat-women-6th-match-wbbl-2026/"

response = requests.get(url, timeout=20)

print("Status code:", response.status_code)
print("HTML size:", len(response.text))

Status code: 200
HTML size: 207601


# **Parse the HTML**

In [4]:
soup = BeautifulSoup(response.text, "html.parser")

print("Page title:", soup.title.get_text(strip=True) if soup.title else "Not found")

match = soup.select_one("main#main")

if match:
    print("Match ID:", match.get("data-id"))
    print("Match phase:", match.get("data-phase"))
    print("Teams:", match.get("data-teams"))
else:
    print("Match container not found")

Page title: Sydney Thunder Women vs Brisbane Heat Women Scorecard & Commentary, 6th Match | WBBL 2026 | CricketSix.com
Match ID: 162733
Match phase: complete
Teams: 362,356


# **Locate the complete commentary section**

In [5]:
commentary = soup.select_one("section.comm-sec")

if commentary:
    print("Commentary section found!")

    all_events = commentary.select("li")

    print("Total commentary events:", len(all_events))
    print("\nFirst 10 events:\n")

    for i, event in enumerate(all_events[:10]):
        print(
            i,
            "|",
            event.get("class"),
            "|",
            event.get_text(" ", strip=True)[:180]
        )
else:
    print("Commentary section not found")

Commentary section found!
Total commentary events: 246

First 10 events:

0 | ['ball', 'ball-b'] | 16.4 4 Lily Bassingthwaighte to Nadine de Klerk, FOUR, Nadine de Klerk crashes it away to sweeper cover, and that's racing all the way to the rope for four!
1 | ['ball', 'ball-note'] | Nadine de Klerk comes in, a right-handed bat.
2 | ['ball', 'ball-w'] | 16.3 W Lily Bassingthwaighte to Laura Harris, out Caught by Grace Harris!! What a stunning catch, and it's a big wicket too, Laura Harris is caught by Grace Harris off Lily Bassing
3 | ['ball', 'ball-note'] | Wicket, caught.
4 | ['ball'] | 16.3 Lily Bassingthwaighte to Laura Harris, wide.
5 | ['ball', 'ball-b'] | 16.2 4 Lily Bassingthwaighte to Laura Harris, FOUR, Laura Harris pulls it away to deep square leg, and it races across the rope for four!
6 | ['ball'] | 16.1 Lily Bassingthwaighte to Heather Knight, 1 run, Heather Knight works this out to deep backward point to pick up a single.
7 | ['ball', 'ball-note'] | Lily Bassingthwaighte 

## **Inspect the innings and over structure**

In [6]:
print("DIRECT CHILDREN OF COMMENTARY SECTION:\n")

for child in commentary.children:
    if getattr(child, "name", None):
        print(
            "Tag:", child.name,
            "| Class:", child.get("class"),
            "| ID:", child.get("id"),
            "| Text:", child.get_text(" ", strip=True)[:180]
        )

DIRECT CHILDREN OF COMMENTARY SECTION:

Tag: h2 | Class: None | ID: h-comm | Text: Ball-by-ball commentary
Tag: div | Class: ['filters', 'comm-filters'] | ID: None | Text: All Wickets 4s & 6s Overs
Tag: p | Class: ['empty'] | ID: None | Text: 
Tag: ol | Class: ['comm'] | ID: commentary | Text: 16.4 4 Lily Bassingthwaighte to Nadine de Klerk, FOUR, Nadine de Klerk crashes it away to sweeper cover, and that's racing all the way to the rope for four! Nadine de Klerk comes i
Tag: button | Class: ['btn'] | ID: None | Text: Load older commentary


In [7]:
end_marker = commentary.find(
    string=lambda text: text and "End of over 20" in text
)

if end_marker:
    print(end_marker.parent.parent.prettify()[:5000])
else:
    print("End-of-over marker not found")

<div class="oe-h">
 <strong>
  End of over 20
 </strong>
 <span>
  10 runs · 178-5
 </span>
</div>



# **Inspect every event type**

In [8]:
from collections import Counter

event_types = Counter()

for event in commentary.select("#commentary > li"):
    classes = event.get("class", [])

    if "over-end" in classes:
        event_types["over-end"] += 1
    elif "ball-note" in classes:
        event_types["ball-note"] += 1
    else:
        event_types["delivery"] += 1

print("Event counts:\n")

for event_type, count in event_types.items():
    print(event_type, ":", count)

print("\nTotal:", sum(event_types.values()))

Event counts:

delivery : 178
ball-note : 40
over-end : 28

Total: 246


# **Inspect the event HTML structure**

In [9]:
events = commentary.select("#commentary > li")

class_counts = Counter()

for event in events:
    classes = event.get("class", [])

    if not classes:
        class_counts["no class"] += 1
    else:
        for class_name in classes:
            class_counts[class_name] += 1

print("Discovered HTML classes:\n")

for class_name, count in class_counts.items():
    print(f"{class_name}: {count}")

Discovered HTML classes:

ball: 218
ball-b: 43
ball-note: 40
ball-w: 8
over-end: 28
ball-6: 10


# **Inspect one example of every unique class combination**

In [10]:
seen = set()

for event in events:
    classes = tuple(sorted(event.get("class", [])))

    if classes not in seen:
        seen.add(classes)

        print("\nCLASS COMBINATION:", classes)
        print(event.prettify()[:2000])


CLASS COMBINATION: ('ball', 'ball-b')
<li class="ball ball-b" data-ts="1793518334461">
 <span class="ball-over">
  16.4
  <span class="bb bb-b bb-4" title="Four">
   4
  </span>
 </span>
 <div class="ball-text">
  <p>
   Lily Bassingthwaighte to Nadine de Klerk, FOUR, Nadine de Klerk crashes it away to sweeper cover, and that's racing all the way to the rope for four!
  </p>
 </div>
</li>


CLASS COMBINATION: ('ball', 'ball-note')
<li class="ball ball-note" data-ts="1793518310164">
 <p>
  Nadine de Klerk comes in, a right-handed bat.
 </p>
</li>


CLASS COMBINATION: ('ball', 'ball-w')
<li class="ball ball-w" data-ts="1793518234543">
 <span class="ball-over">
  16.3
  <span class="bb bb-w" title="Wicket">
   W
  </span>
 </span>
 <div class="ball-text">
  <p>
   Lily Bassingthwaighte to Laura Harris, out Caught by Grace Harris!! What a stunning catch, and it's a big wicket too, Laura Harris is caught by Grace Harris off Lily Bassingthwaighte, a well-made 35 from just 16 balls, with 6 f

# **Build a structured event dataset**

In [11]:
events = commentary.select("#commentary > li")

event_records = []

for position, event in enumerate(events):

    classes = event.get("class", [])

    record = {
        "position": position,
        "event_type": None,
        "ball_number": None,
        "indicator": None,
        "over_number": None,
        "score": None,
        "commentary": None,
        "data_ts": event.get("data-ts")
    }

    if "over-end" in classes:
        record["event_type"] = "over_summary"

        heading = event.select_one(".oe-h strong")
        score = event.select_one(".oe-h span")
        players = event.select_one(".over-who")

        record["over_number"] = (
            re.search(r"\d+", heading.get_text(" ", strip=True)).group()
            if heading else None
        )

        record["score"] = (
            score.get_text(" ", strip=True) if score else None
        )

        record["commentary"] = (
            players.get_text(" ", strip=True) if players else None
        )

    elif "ball-note" in classes:
        record["event_type"] = "note"

        note = event.select_one("p")
        record["commentary"] = (
            note.get_text(" ", strip=True) if note else None
        )

    else:
        record["event_type"] = "delivery"

        ball_element = event.select_one(".ball-over")

        if ball_element:
            record["ball_number"] = (
                ball_element.contents[0].strip()
            )

            indicator_element = ball_element.select_one(".bb")

            if indicator_element:
                record["indicator"] = (
                    indicator_element.get_text(strip=True)
                )

        text_element = event.select_one(".ball-text p")

        record["commentary"] = (
            text_element.get_text(" ", strip=True)
            if text_element else None
        )

    event_records.append(record)

events_df = pd.DataFrame(event_records)

print("Total events:", len(events_df))
print("\nEvent type counts:")
print(events_df["event_type"].value_counts())

print("\nFirst 12 events:")
print(events_df.head(12).to_string(index=False))

Total events: 246

Event type counts:
event_type
delivery        178
note             40
over_summary     28
Name: count, dtype: int64

First 12 events:
 position   event_type ball_number indicator over_number           score                                                                                                                                                                                                                                                                                                                                commentary       data_ts
        0     delivery        16.4         4        None            None                                                                                                                                                                                     Lily Bassingthwaighte to Nadine de Klerk, FOUR, Nadine de Klerk crashes it away to sweeper cover, and that's racing all the way to the rope for four! 1793518334461
        1    

## **Extract over numbers and ball numbers**
- Add two columns to events_df:

In [12]:
events_df["over"] = pd.to_numeric(
    events_df["ball_number"].str.split(".").str[0],
    errors="coerce"
)

events_df["ball"] = pd.to_numeric(
    events_df["ball_number"].str.split(".").str[1],
    errors="coerce"
)

print(
    events_df[
        ["position", "event_type", "ball_number", "over", "ball", "score"]
    ].head(15).to_string(index=False)
)

 position   event_type ball_number  over  ball           score
        0     delivery        16.4  16.0   4.0            None
        1         note        None   NaN   NaN            None
        2     delivery        16.3  16.0   3.0            None
        3         note        None   NaN   NaN            None
        4     delivery        16.3  16.0   3.0            None
        5     delivery        16.2  16.0   2.0            None
        6     delivery        16.1  16.0   1.0            None
        7         note        None   NaN   NaN            None
        8 over_summary        None   NaN   NaN 14 runs · 170-3
        9     delivery        15.6  15.0   6.0            None
       10     delivery        15.5  15.0   5.0            None
       11     delivery        15.4  15.0   4.0            None
       12     delivery        15.3  15.0   3.0            None
       13     delivery        15.2  15.0   2.0            None
       14     delivery        15.1  15.0   1.0         

In [13]:
over_summaries = events_df[
    events_df["event_type"] == "over_summary"
]

print(
    over_summaries[
        ["position", "over_number", "score", "commentary"]
    ].to_string(index=False)
)

 position over_number           score                                                                      commentary
        8          16 14 runs · 170-3              Heather Knight 53(28) · Laura Harris 31(14) · Sarah Glenn 3-0-40-0
       16          15 16 runs · 156-3            Laura Harris 26(11) · Heather Knight 44(25) · Lucy Hamilton 3-0-34-1
       23          14 20 runs · 140-3             Laura Harris 14(7) · Heather Knight 40(23) · Sianna Ginger 3-0-33-0
       31          13  9 runs · 120-3              Laura Harris 2(3) · Heather Knight 33(21) · Lucy Hamilton 2-0-18-1
       42          12  7 runs · 111-2           Anika Learoyd 31(25) · Heather Knight 33(21) · Sianna Ginger 2-0-13-0
       50          11 15 runs · 104-2        Heather Knight 31(19) · Anika Learoyd 26(21) · Orla Prendergast 2-0-23-1
       59          10  14 runs · 89-2            Heather Knight 17(14) · Anika Learoyd 25(20) · Charli Knott 2-0-19-0
       66           9   6 runs · 75-2             Heathe

## **Create chronological delivery data**

In [14]:
# Assign innings using the delivery sequence

deliveries_original = events_df[
    events_df["event_type"] == "delivery"
].copy()

deliveries_original = deliveries_original.sort_values("position")

# Detect the transition from innings 1 to innings 2
over_changes = deliveries_original["over"].diff()

transition = deliveries_original[over_changes > 0]

if transition.empty:
    raise ValueError("Innings transition not found.")

boundary = transition.iloc[0]["position"]

# Add innings labels to the complete events dataframe
events_df["innings"] = (
    events_df["position"] >= boundary
).astype(int) + 1

# Extract deliveries again with innings labels
deliveries_df = events_df[
    events_df["event_type"] == "delivery"
].copy()

# Chronological order
deliveries_df = deliveries_df.sort_values(
    by=["innings", "over", "ball", "position"],
    ascending=[True, True, True, False]
).reset_index(drop=True)

print("Boundary position:", boundary)
print("\nDeliveries per innings:")
print(deliveries_df.groupby("innings").size())

print("\nFirst 15 deliveries:")
print(
    deliveries_df[
        ["innings", "ball_number", "commentary"]
    ].head(15).to_string(index=False)
)

print("\nLast 15 deliveries:")
print(
    deliveries_df[
        ["innings", "ball_number", "commentary"]
    ].tail(15).to_string(index=False)
)

Boundary position: 144

Deliveries per innings:
innings
1    107
2     71
dtype: int64

First 15 deliveries:
 innings ball_number                                                                                                                                                commentary
       1         0.1                                                                                                                 Lucy Hamilton to Phoebe Litchfield, wide.
       1         0.1                                                                                                               Lucy Hamilton to Phoebe Litchfield, no run.
       1         0.2                                                                                                                 Lucy Hamilton to Phoebe Litchfield, wide.
       1         0.2                      Lucy Hamilton to Phoebe Litchfield, FOUR, Phoebe Litchfield threads this through the covers and races away to the boundary for four!
       1        

## **Extract over summaries separately**

In [15]:
# Extract over summaries from the raw event dataset

overs_df = events_df[
    events_df["event_type"] == "over_summary"
].copy()

# Convert over number to numeric
overs_df["over_number"] = pd.to_numeric(
    overs_df["over_number"],
    errors="coerce"
)

# Extract runs scored in the over
overs_df["over_runs"] = pd.to_numeric(
    overs_df["score"].str.extract(r"(\d+)\s+runs?")[0],
    errors="coerce"
)

# Extract cumulative score and wickets
score_parts = overs_df["score"].str.extract(
    r"(\d+)-(\d+)"
)

overs_df["cumulative_runs"] = pd.to_numeric(
    score_parts[0], errors="coerce"
)

overs_df["wickets"] = pd.to_numeric(
    score_parts[1], errors="coerce"
)

# Chronological order
overs_df = overs_df.sort_values(
    ["innings", "over_number"]
).reset_index(drop=True)

print("Over summaries:", len(overs_df))

print(
    overs_df[
        [
            "innings",
            "over_number",
            "over_runs",
            "cumulative_runs",
            "wickets"
        ]
    ].to_string(index=False)
)

Over summaries: 28
 innings  over_number  over_runs  cumulative_runs  wickets
       1            1          9                9        0
       1            2          3               12        0
       1            3          8               20        1
       1            4          5               25        1
       1            5         16               41        1
       1            6         13               54        2
       1            7         10               64        2
       1            8          5               69        2
       1            9          6               75        2
       1           10         14               89        2
       1           11         15              104        2
       1           12          7              111        2
       1           13          9              120        3
       1           14         20              140        3
       1           15         16              156        3
       1           16         14     